# Vaelo — Valuation Report Generation Pipeline

This is Pipeline 1 of 3 (Valuation Report). It is **deterministic and formula-driven** —
no LLM, no trained model, by design. Every number in the output must trace back to a
formula a Chartered Accountant can defend to their client. That auditability is Vaelo's
core differentiator (see SRS Section 2.4 and Section 11).

**Pipeline stages:**
1. Required documents & data (what a CA must submit)
2. Structured intake schema (how that data is represented in code)
3. Calculation engine (the actual DCF math)
4. Comparable-multiple cross-check (sanity check against industry multiples)
5. Report generator (templated text output, not generative AI)
6. End-to-end example run

## 1. Required Documents & Data

Before running this pipeline for a real client, the CA must supply:

| Item | Why it's needed |
|---|---|
| 3–5 years historical P&L (Revenue, EBITDA, EBIT, D&A) | Basis for projecting forward cash flows |
| Latest Balance Sheet (Total Debt, Cash & Equivalents) | Needed to go from Enterprise Value → Equity Value |
| Applicable tax rate | Used in the FCF formula |
| Sector / industry classification | Used for the comparable-multiple cross-check |
| Growth, CapEx, and working-capital assumptions | Either CA-provided, or a documented default with the CA's sign-off |

If any of these are missing or the CA can't provide clean historicals, that's a real
intake problem worth flagging back — this pipeline should not silently guess at
missing data.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import date

## 2. Structured Intake Schema

`ValuationRequest` is the single object a CA "submits" for a valuation report —
this is the boundary between the CA and the engine. In v1 this gets filled in
manually (Colab cell, form, or spreadsheet import later); no document-parsing
model is involved yet, per the founder-in-the-loop scope for v1.

In [ ]:
@dataclass
class ClientMeta:
    """Who this report is for and why — SRS FR-1.1"""
    client_name: str
    ca_firm_name: str
    trigger_event: str          # 'loan', 'sale', 'merger', 'investor_pitch', 'periodic'
    report_date: str = field(default_factory=lambda: date.today().isoformat())
    sector: str = "General"


@dataclass
class FinancialInputs:
    """Raw financial data submitted for the client — SRS FR-1.2"""
    historical_revenue: List[float]      # most recent years, oldest to newest, in Cr
    historical_ebitda: List[float]
    current_ebit: float
    current_da: float                     # depreciation & amortization
    current_capex: float
    current_nwc_change: float             # change in net working capital
    tax_rate: float                       # e.g. 0.25 for 25%
    net_debt: float                       # total debt - cash & equivalents


@dataclass
class WACCInputs:
    """Build-up method inputs — no public beta exists for a private SME,
    so CAPM alone doesn't apply. See the earlier guide for sourcing each figure."""
    risk_free_rate: float
    equity_risk_premium: float
    size_premium: float
    company_specific_premium: float
    cost_of_debt: float
    equity_weight: float                  # E / (E+D)
    debt_weight: float                    # D / (E+D)
    tax_rate: float

    def cost_of_equity(self) -> float:
        return (
            self.risk_free_rate
            + self.equity_risk_premium
            + self.size_premium
            + self.company_specific_premium
        )

    def wacc(self) -> float:
        ke = self.cost_of_equity()
        after_tax_kd = self.cost_of_debt * (1 - self.tax_rate)
        return (self.equity_weight * ke) + (self.debt_weight * after_tax_kd)


@dataclass
class ProjectionAssumptions:
    """Forward-looking assumptions — CA-adjustable, SRS FR-2.4"""
    projection_years: int
    revenue_growth_rates: List[float]      # one per projection year
    ebitda_margin: float
    da_as_pct_revenue: float
    capex_as_pct_revenue: float
    nwc_change_as_pct_revenue: float
    terminal_growth_rate: float            # must be < WACC


@dataclass
class ComparableAssumptions:
    """For the cross-check in Section 4 — sourced per-sector (e.g. Damodaran data,
    Indian sector reports). This is a sanity check, not the primary valuation method."""
    ev_ebitda_multiple_low: float
    ev_ebitda_multiple_high: float


@dataclass
class ValuationRequest:
    """The single object a CA submits — the intake boundary for this pipeline."""
    meta: ClientMeta
    financials: FinancialInputs
    wacc_inputs: WACCInputs
    assumptions: ProjectionAssumptions
    comparables: ComparableAssumptions

## 3. Calculation Engine (Deterministic DCF)

This is the core math — identical logic to the standalone script from the earlier
guide, now organized to consume a `ValuationRequest` directly.

In [ ]:
def project_free_cash_flows(base_revenue: float, req: ValuationRequest) -> List[float]:
    """Project FCF for each year of the projection period."""
    fcfs = []
    revenue = base_revenue
    inputs = req.financials
    assumptions = req.assumptions

    for growth_rate in assumptions.revenue_growth_rates:
        revenue = revenue * (1 + growth_rate)
        ebitda = revenue * assumptions.ebitda_margin
        da = revenue * assumptions.da_as_pct_revenue
        ebit = ebitda - da
        capex = revenue * assumptions.capex_as_pct_revenue
        nwc_change = revenue * assumptions.nwc_change_as_pct_revenue

        fcf = ebit * (1 - inputs.tax_rate) + da - capex - nwc_change
        fcfs.append(fcf)

    return fcfs


def discount_cash_flows(fcfs: List[float], wacc: float) -> List[float]:
    """Return present value of each projected FCF."""
    return [fcf / ((1 + wacc) ** (t + 1)) for t, fcf in enumerate(fcfs)]


def terminal_value(final_year_fcf: float, wacc: float, terminal_growth: float) -> float:
    """Gordon Growth terminal value. Hard constraint: g must be < WACC."""
    if terminal_growth >= wacc:
        raise ValueError(
            f"Terminal growth rate ({terminal_growth:.1%}) must be less than "
            f"WACC ({wacc:.1%}) — a terminal growth rate above the discount rate "
            f"implies infinite value, which is not a valid result."
        )
    return final_year_fcf * (1 + terminal_growth) / (wacc - terminal_growth)


def calculate_enterprise_value(base_revenue: float, req: ValuationRequest) -> dict:
    """The full DCF calculation. Returns every intermediate figure, not just
    the final number — auditability is the entire point of this pipeline."""

    wacc = req.wacc_inputs.wacc()

    fcfs = project_free_cash_flows(base_revenue, req)
    pv_fcfs = discount_cash_flows(fcfs, wacc)

    tv = terminal_value(fcfs[-1], wacc, req.assumptions.terminal_growth_rate)
    pv_tv = tv / ((1 + wacc) ** req.assumptions.projection_years)

    enterprise_value = sum(pv_fcfs) + pv_tv
    equity_value = enterprise_value - req.financials.net_debt

    return {
        "wacc": wacc,
        "cost_of_equity": req.wacc_inputs.cost_of_equity(),
        "projected_fcfs": fcfs,
        "present_value_fcfs": pv_fcfs,
        "terminal_value": tv,
        "pv_terminal_value": pv_tv,
        "enterprise_value": enterprise_value,
        "net_debt": req.financials.net_debt,
        "equity_value": equity_value,
    }


def sensitivity_grid(
    base_revenue: float,
    req: ValuationRequest,
    wacc_range: List[float],
    growth_range: List[float],
) -> dict:
    """Build a WACC x terminal-growth sensitivity matrix — SRS FR-2.2.
    Shows how the valuation moves as assumptions change, rather than
    presenting a single number as gospel."""

    grid = {}
    for w in wacc_range:
        row = {}
        for g in growth_range:
            fcfs = project_free_cash_flows(base_revenue, req)
            pv_fcfs = discount_cash_flows(fcfs, w)
            tv = terminal_value(fcfs[-1], w, g)
            pv_tv = tv / ((1 + w) ** req.assumptions.projection_years)
            ev = sum(pv_fcfs) + pv_tv
            row[f"g={g:.1%}"] = round(ev, 2)
        grid[f"WACC={w:.1%}"] = row

    return grid

## 4. Comparable-Multiple Cross-Check

SRS FR-2.3: apply an industry EV/EBITDA multiple as a sanity check against the DCF
output. If the two methods disagree by more than ~35%, that's a signal to
re-examine the assumptions before finalizing — not something to paper over.

In [ ]:
def comparable_cross_check(latest_ebitda: float, req: ValuationRequest, dcf_ev: float) -> dict:
    low = latest_ebitda * req.comparables.ev_ebitda_multiple_low
    high = latest_ebitda * req.comparables.ev_ebitda_multiple_high
    midpoint = (low + high) / 2

    deviation = abs(dcf_ev - midpoint) / midpoint if midpoint else 0
    flag = deviation > 0.35

    return {
        "multiple_low_ev": round(low, 2),
        "multiple_high_ev": round(high, 2),
        "multiple_midpoint_ev": round(midpoint, 2),
        "dcf_vs_multiple_deviation_pct": round(deviation * 100, 1),
        "flagged_for_review": flag,
    }

## 5. Report Generator (Templated, Not Generative)

SRS FR-5.3: narrative sections use parameterized templates, not free-form
generative text — every sentence traces to a computed value. This function
takes the raw engine output and formats it into a document a CA can review
and hand to their client.

In [ ]:
def build_valuation_report(
    req: ValuationRequest,
    dcf_result: dict,
    sensitivity: dict,
    cross_check: dict,
) -> str:
    """Templated report generation — plain string formatting, no LLM."""

    lines = []
    lines.append(f"VALUATION REPORT — {req.meta.client_name}")
    lines.append(f"Prepared for: {req.meta.ca_firm_name}")
    lines.append(f"Date: {req.meta.report_date}")
    lines.append(f"Trigger event: {req.meta.trigger_event}")
    lines.append(f"Sector: {req.meta.sector}")
    lines.append("=" * 60)

    lines.append("\n--- DISCOUNT RATE (WACC) BUILD-UP ---")
    lines.append(f"Cost of Equity:       {dcf_result['cost_of_equity']:.2%}")
    lines.append(f"WACC:                 {dcf_result['wacc']:.2%}")

    lines.append("\n--- PROJECTED FREE CASH FLOWS (Rs. Cr) ---")
    for i, (fcf, pv) in enumerate(zip(dcf_result["projected_fcfs"], dcf_result["present_value_fcfs"]), start=1):
        lines.append(f"Year {i}: FCF = {fcf:.2f}  |  Present Value = {pv:.2f}")

    lines.append("\n--- TERMINAL VALUE ---")
    lines.append(f"Terminal Value:       Rs. {dcf_result['terminal_value']:.2f} Cr")
    lines.append(f"PV of Terminal Value: Rs. {dcf_result['pv_terminal_value']:.2f} Cr")

    lines.append("\n--- VALUATION SUMMARY ---")
    lines.append(f"Enterprise Value:     Rs. {dcf_result['enterprise_value']:.2f} Cr")
    lines.append(f"Less: Net Debt:       Rs. {dcf_result['net_debt']:.2f} Cr")
    lines.append(f"Equity Value:         Rs. {dcf_result['equity_value']:.2f} Cr")

    lines.append("\n--- COMPARABLE-MULTIPLE CROSS-CHECK ---")
    lines.append(f"Implied EV range (multiple method): Rs. {cross_check['multiple_low_ev']:.2f} Cr - Rs. {cross_check['multiple_high_ev']:.2f} Cr")
    lines.append(f"DCF vs. multiple deviation:          {cross_check['dcf_vs_multiple_deviation_pct']:.1f}%")
    if cross_check["flagged_for_review"]:
        lines.append("FLAG: DCF and multiple-based estimates diverge by more than 35%.")
        lines.append("      Review assumptions before presenting this report to the client.")
    else:
        lines.append("DCF and multiple-based estimates are broadly consistent.")

    lines.append("\n--- SENSITIVITY ANALYSIS (Enterprise Value, Rs. Cr) ---")
    header = "WACC \\ g".ljust(12) + "".join(
        g.ljust(12) for g in list(sensitivity.values())[0].keys()
    )
    lines.append(header)
    for wacc_label, row in sensitivity.items():
        line = wacc_label.ljust(12) + "".join(str(v).ljust(12) for v in row.values())
        lines.append(line)

    lines.append("\n" + "=" * 60)
    lines.append("This report is formula-driven and fully auditable. Every figure above")
    lines.append("traces to the assumptions and inputs supplied for this engagement.")
    lines.append("Prepared via Vaelo — for review by the submitting CA before client delivery.")

    return "\n".join(lines)

## 6. End-to-End Example Run

Replace the values below with real client data before using this for an actual
engagement. This example uses illustrative figures only.

In [ ]:
# --- Example ValuationRequest (replace with real client data) ---

example_request = ValuationRequest(
    meta=ClientMeta(
        client_name="Example SME Pvt Ltd",
        ca_firm_name="Example & Associates",
        trigger_event="loan",
        sector="Manufacturing",
    ),
    financials=FinancialInputs(
        historical_revenue=[8.0, 9.2, 10.5],
        historical_ebitda=[1.2, 1.5, 1.8],
        current_ebit=1.4,
        current_da=0.4,
        current_capex=0.5,
        current_nwc_change=0.2,
        tax_rate=0.25,
        net_debt=1.5,
    ),
    wacc_inputs=WACCInputs(
        risk_free_rate=0.071,          # current India 10Y G-Sec — update per report
        equity_risk_premium=0.065,     # e.g. Damodaran India ERP — update per report
        size_premium=0.05,
        company_specific_premium=0.02,
        cost_of_debt=0.11,
        equity_weight=0.7,
        debt_weight=0.3,
        tax_rate=0.25,
    ),
    assumptions=ProjectionAssumptions(
        projection_years=5,
        revenue_growth_rates=[0.12, 0.10, 0.09, 0.08, 0.07],
        ebitda_margin=0.18,
        da_as_pct_revenue=0.04,
        capex_as_pct_revenue=0.05,
        nwc_change_as_pct_revenue=0.02,
        terminal_growth_rate=0.04,
    ),
    comparables=ComparableAssumptions(
        ev_ebitda_multiple_low=5.5,
        ev_ebitda_multiple_high=7.0,
    ),
)

# --- Run the pipeline ---

base_revenue = example_request.financials.historical_revenue[-1]

dcf_result = calculate_enterprise_value(base_revenue, example_request)

sensitivity = sensitivity_grid(
    base_revenue,
    example_request,
    wacc_range=[dcf_result["wacc"] - 0.02, dcf_result["wacc"], dcf_result["wacc"] + 0.02],
    growth_range=[
        example_request.assumptions.terminal_growth_rate - 0.01,
        example_request.assumptions.terminal_growth_rate,
        example_request.assumptions.terminal_growth_rate + 0.01,
    ],
)

cross_check = comparable_cross_check(
    example_request.financials.historical_ebitda[-1],
    example_request,
    dcf_result["enterprise_value"],
)

report = build_valuation_report(example_request, dcf_result, sensitivity, cross_check)
print(report)

## Next Steps

- **Sanity-check the output** against Section 6 of the earlier guide (terminal
  growth < WACC — already enforced above; WACC in a plausible 12–18% range for
  an Indian SME; cross-check flag not triggered, or investigated if it is).
- **Replace placeholder WACC inputs** with real current market data before using
  this for an actual client (10Y G-Sec yield and equity risk premium both move
  and should be sourced fresh per report, not hardcoded).
- **This is intentionally manual for v1** — per SRS FR-6, founder-in-the-loop
  review is expected at this stage. Don't rush to automate intake before the
  underlying model has been stress-tested against a few real engagements.
- Once this pipeline is validated on real data, the next step is wrapping
  `build_valuation_report` output into a formatted PDF (separate task) — the
  calculation and report-text logic above stays exactly the same either way.